# Lecture 04 — A static crawler, end to end

> *"At some point you stop following tutorials and just build the thing."*

In Lectures 02 and 03 you learned to fetch and to parse. In this lecture we put both together to build a complete, runnable crawler that walks an entire site, persists what it finds, and can resume after a crash.

## What you'll be able to do after this lecture

- Walk a paginated site from the first page to the last.
- Deduplicate URLs you've already seen.
- Persist results in two formats: JSONL for streaming + SQLite for queries.
- Resume a partially-completed crawl without re-fetching everything.
- Handle errors per-page without killing the whole crawl.

## Target

We'll crawl [`books.toscrape.com`](https://books.toscrape.com/) and build a database of every book on the site (~1000 books, 50 pages of listings, plus one detail page each). It's the "hello world" of static-site crawling.


## 1. Anatomy of the crawl

Five steps from Lecture 01, made concrete:

```
seed:    https://books.toscrape.com/catalogue/page-1.html
fetch:   GET each listing page; on the last page, the "next" link disappears.
parse:   extract product cards; for each card, also extract its detail-page URL.
extract: from each detail page, extract title, price, in-stock, rating, description.
store:   one row per book in SQLite; one line per book in books.jsonl.
```

The work queue evolves like this:

```
[page-1.html]                                            # seed
[page-1.html, ...20 detail urls...]                      # after parsing page 1
[page-2.html, ...20 detail urls...]                      # after fetching all the details
[page-2.html, ...20 detail urls..., page-3.html]
...
```

This is BFS, basically. There's a queue, you take the next URL off, you fetch it, you may add new URLs to the queue. Stop when the queue is empty.


## 2. Pagination — find the pattern, exploit it

Most paginated sites have one of two URL patterns:

- `?page=2` (query string)
- `/page-2.html` or `/page/2/` (path)

`books.toscrape.com` uses path-based pagination: `catalogue/page-1.html`, `catalogue/page-2.html`, ..., `catalogue/page-50.html`. We *could* hard-code 50, but a good crawler doesn't assume — it follows the "next" link until it disappears.


In [ ]:
import httpx
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE = "https://books.toscrape.com/"

def find_next_page(soup: BeautifulSoup, current_url: str) -> str | None:
    next_li = soup.select_one("li.next a")
    if next_li is None:
        return None
    return urljoin(current_url, next_li["href"])

# Try it on page 1
r = httpx.get(urljoin(BASE, "catalogue/page-1.html"), timeout=10.0)
soup = BeautifulSoup(r.text, "lxml")
print("next page:", find_next_page(soup, str(r.url)))


`urljoin` matters. Some sites give you absolute URLs in `href`, some give you relative URLs (`../page-2.html`). `urljoin(base, href)` handles both correctly. Hand-concatenating strings will eventually bite you.

## 3. Extracting listing pages and detail URLs

In [ ]:
def extract_products_from_listing(soup: BeautifulSoup, page_url: str) -> list[dict]:
    products = []
    for card in soup.select("article.product_pod"):
        a = card.select_one("h3 a")
        if a is None:
            continue
        detail_href = urljoin(page_url, a["href"])
        products.append({
            "title": a["title"],
            "detail_url": detail_href,
            "listing_price": card.select_one(".price_color").get_text(strip=True),
        })
    return products


products = extract_products_from_listing(soup, str(r.url))
for p in products[:3]:
    print(p)


## 4. Extracting a detail page

The detail page has more fields than the listing card. Let's pull what's worth pulling.

In [ ]:
def extract_detail(soup: BeautifulSoup, url: str) -> dict:
    title = soup.select_one("div.product_main h1").get_text(strip=True)
    price = soup.select_one("p.price_color").get_text(strip=True)
    in_stock = soup.select_one("p.availability").get_text(strip=True)

    rating_class = soup.select_one("p.star-rating").get("class", [])
    rating = next((c for c in rating_class if c != "star-rating"), None)

    description_el = soup.select_one("#product_description ~ p")
    description = description_el.get_text(strip=True) if description_el else None

    # The product info table holds UPC, type, etc.
    table_rows = soup.select("table.table.table-striped tr")
    info = {}
    for row in table_rows:
        key = row.find("th").get_text(strip=True).lower()
        val = row.find("td").get_text(strip=True)
        info[key] = val

    return {
        "url": url,
        "title": title,
        "price": price,
        "in_stock": in_stock,
        "rating": rating,
        "description": description,
        "upc": info.get("upc"),
        "category": (soup.select_one("ul.breadcrumb li:nth-child(3) a") or {}).get_text(strip=True) if soup.select_one("ul.breadcrumb li:nth-child(3) a") else None,
    }


detail_url = products[0]["detail_url"]
dr = httpx.get(detail_url, timeout=10.0)
detail_soup = BeautifulSoup(dr.text, "lxml")
print(extract_detail(detail_soup, detail_url))


## 5. Persistence — JSONL + SQLite

You have a choice. JSONL is great for streaming and pipes; SQLite is great for queries. We'll do both, because it costs almost nothing and earns you a lot.

**JSONL** = one JSON object per line. Append-only. Easy to grep, easy to load with pandas, easy to ship.

**SQLite** = a real relational database in a single file. Great when you want to ask questions like *"all books with rating Five and price < £20"*.

In [ ]:
import json
import sqlite3
from pathlib import Path

DATA_DIR = Path("./out")
DATA_DIR.mkdir(exist_ok=True)
JSONL = DATA_DIR / "books.jsonl"
DB = DATA_DIR / "books.db"


def init_db(conn: sqlite3.Connection):
    conn.execute("""
        CREATE TABLE IF NOT EXISTS books (
            url TEXT PRIMARY KEY,
            title TEXT NOT NULL,
            price TEXT,
            in_stock TEXT,
            rating TEXT,
            description TEXT,
            upc TEXT,
            category TEXT,
            fetched_at TEXT NOT NULL
        )
    """)
    conn.commit()


def save(record: dict, conn: sqlite3.Connection, jsonl_path: Path):
    # JSONL append
    with jsonl_path.open("a") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
    # SQLite upsert
    conn.execute(
        """INSERT OR REPLACE INTO books
           (url, title, price, in_stock, rating, description, upc, category, fetched_at)
           VALUES (?, ?, ?, ?, ?, ?, ?, ?, datetime('now'))""",
        (record["url"], record["title"], record["price"], record["in_stock"],
         record["rating"], record["description"], record["upc"], record["category"]),
    )
    conn.commit()


**Why `INSERT OR REPLACE`?** It's the simplest form of *upsert*: if we re-fetch a URL we already have, the new row replaces the old one. That's what we want — fresher data wins.

**Why `url` as primary key?** Because it's the one field we know is unique. Site-specific IDs (UPC) might be missing on some pages.

## 6. Resumability — knowing what you've already done

If your crawler crashes after 800 of 1000 books, you don't want to start over. The cheapest way: query the DB for "URLs we've already saved" and skip them in the queue.

In [ ]:
def already_done(conn: sqlite3.Connection) -> set[str]:
    cur = conn.execute("SELECT url FROM books")
    return {row[0] for row in cur.fetchall()}


## 7. The full crawler

Here it is, end to end. Read it slowly — every line is doing real work.

In [ ]:
import time, random, logging
from collections import deque

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(message)s")
log = logging.getLogger("crawler")

HEADERS = {
    "User-Agent": "CrawlingTutorial/0.1 (+https://github.com/Vladimir-125/CrawlingTutorial)",
    "Accept-Language": "en-US,en;q=0.9",
}

def fetch(client: httpx.Client, url: str, max_attempts: int = 4) -> httpx.Response | None:
    for attempt in range(max_attempts):
        try:
            r = client.get(url)
        except httpx.RequestError as e:
            log.warning(f"  network error on {url}: {e}")
        else:
            if r.status_code == 200:
                return r
            if r.status_code in (404, 410):
                log.warning(f"  permanent {r.status_code} on {url}; skipping")
                return None
            log.warning(f"  transient {r.status_code} on {url}")
        time.sleep((2 ** attempt) + random.random())
    return None


def crawl():
    conn = sqlite3.connect(DB)
    init_db(conn)
    done = already_done(conn)
    log.info(f"starting; {len(done)} URLs already in DB")

    queue = deque([urljoin(BASE, "catalogue/page-1.html")])
    seen_listing = set()

    with httpx.Client(headers=HEADERS, timeout=10.0, follow_redirects=True) as client:
        while queue:
            url = queue.popleft()
            if url in seen_listing:
                continue
            seen_listing.add(url)

            log.info(f"listing: {url}")
            r = fetch(client, url)
            if r is None:
                continue
            soup = BeautifulSoup(r.text, "lxml")

            for product in extract_products_from_listing(soup, url):
                detail_url = product["detail_url"]
                if detail_url in done:
                    continue
                dr = fetch(client, detail_url)
                if dr is None:
                    continue
                detail_soup = BeautifulSoup(dr.text, "lxml")
                try:
                    record = extract_detail(detail_soup, detail_url)
                except Exception as e:
                    log.error(f"  parse failed on {detail_url}: {e}")
                    continue
                save(record, conn, JSONL)
                done.add(detail_url)
                time.sleep(0.5)  # be polite

            next_url = find_next_page(soup, url)
            if next_url:
                queue.append(next_url)

    conn.close()
    log.info(f"done. total in DB: {len(done)}")


# crawl()  # uncomment to run; takes a couple of minutes


### What to notice

- **Two queues, conceptually.** The listing-page queue (`queue`) is BFS. The detail pages we fetch immediately as we discover them, because there's no further graph to walk from a detail page.
- **Every fetch is wrapped.** Errors don't kill the crawl — they're logged and the loop moves on.
- **`time.sleep(0.5)` between detail fetches.** Politeness budget. We'll do better in Lecture 06 (per-host rate limiting), but a fixed sleep is the minimum.
- **Resumable.** Restart the script — it'll pick up where it stopped because of the `done` set.

### Run it

In a fresh terminal:

```bash
cd CrawlingTutorial
python -c "from importlib import import_module; ... "  # or paste this notebook into a script
```

Or just run the cell. After it finishes:

```bash
sqlite3 out/books.db 'SELECT COUNT(*) FROM books'
sqlite3 out/books.db 'SELECT title, price, rating FROM books LIMIT 5'
```


## 8. Inspecting your dataset

In [ ]:
# After running crawl(), poke at the data.
conn = sqlite3.connect(DB)
print("row count:", conn.execute("SELECT COUNT(*) FROM books").fetchone()[0])

print("\nrating distribution:")
for rating, n in conn.execute("SELECT rating, COUNT(*) FROM books GROUP BY rating ORDER BY n DESC"):
    print(f"  {rating}: {n}")

print("\ntop 5 most expensive:")
for title, price in conn.execute("SELECT title, price FROM books ORDER BY CAST(REPLACE(price, '£', '') AS REAL) DESC LIMIT 5"):
    print(f"  {price}  {title}")

conn.close()


## 9. What this crawler still doesn't do

Honest accounting. This crawler is good enough for `books.toscrape.com` and many similar sites. It's not good enough for everything yet:

- **Single-threaded.** ~1000 books × ~600ms each = ~10 minutes. Lecture 06 makes it async.
- **No JavaScript.** A SPA crawler will get an empty page from this code. Lecture 05 fixes that.
- **No anti-bot defense.** A site behind Cloudflare will block this immediately. Lecture 08 covers options.
- **Naive politeness.** A fixed 0.5s sleep is OK for a small site, not for a bigger one. Lecture 06 introduces per-host rate limiting.

But the *shape* of the crawler — the BFS queue, the resumable persistence, the wrapped fetch, the per-page error handling — generalizes to all of those.


## Recap

- The seed-fetch-parse-extract-store loop from Lecture 01 is real code you can write in 80 lines.
- Find the pagination link rather than guessing the page count.
- Persist to JSONL *and* SQLite. JSONL is your durable log; SQLite is your queryable view.
- Make resumability cheap by checking the DB before fetching.
- Wrap every fetch and every parse in error handling. The crawl must survive any single page being broken.

## Exercises

1. Run the crawler. Verify ~1000 books in your DB. How long did it take?
2. Add a `categories` table — one row per category, with a `book_id` foreign key (or a `book_categories` join table). Repopulate it from the existing `books` rows.
3. Modify the crawler to also save the cover image to disk under `out/covers/<upc>.jpg`. Use the `image_url` from the listing card.
4. Add command-line arguments via `argparse` so you can run `python crawl.py --output ./mydir` and `python crawl.py --resume` (skips re-init) vs `--fresh` (deletes DB first).

## Up next

**Lecture 05** — many sites in 2026 don't render content in the HTML you `GET`. We'll use Playwright, the modern descendant of Selenium and PyQt5, to drive a real browser and capture the page after JavaScript has filled it in.
